In [1]:
import time, json, warnings
import numpy as np
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline

P = Path("processed"); OUT = P / "balanced"; OUT.mkdir(exist_ok=True)
X = np.load(P / "X_train.npy"); y = np.load(P / "y_train.npy")
SEED = 42

In [2]:
methods = {
    "random_over":      RandomOverSampler(random_state=SEED),
    "random_under":     RandomUnderSampler(random_state=SEED),
    "smote":            SMOTE(random_state=SEED),
    "borderline_smote": BorderlineSMOTE(random_state=SEED),
    "adasyn":           ADASYN(random_state=SEED),
    "under_then_smote": Pipeline([
        ("u", RandomUnderSampler(sampling_strategy=0.5, random_state=SEED)),
        ("s", SMOTE(random_state=SEED))]),
    "smote_tomek":      SMOTETomek(random_state=SEED),   # slow
    "smote_enn":        SMOTEENN(random_state=SEED),     # slow
    "tomek_links":      TomekLinks(),                    # cleaning only
}

In [3]:
log = []
def record(name, yb, secs):
    r = {"method": name, "rows": int(len(yb)),
         "normal": int((yb == 0).sum()), "attack": int((yb == 1).sum()),
         "attack_share": round(float(yb.mean()), 4), "seconds": round(secs, 1)}
    log.append(r); print(r)

np.savez_compressed(OUT / "none.npz", X=X, y=y); record("none", y, 0)

w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
json.dump({"0": float(w[0]), "1": float(w[1])}, open(OUT / "class_weight.json", "w"))
print("class weights:", w)

{'method': 'none', 'rows': 140311, 'normal': 19027, 'attack': 121284, 'attack_share': 0.8644, 'seconds': 0}
class weights: [3.6871551  0.57843986]


In [4]:
for name, m in methods.items():
    t = time.time()
    try:
        Xb, yb = m.fit_resample(X, y)
        np.savez_compressed(OUT / f"{name}.npz",
                            X=Xb.astype(np.float32), y=yb.astype(np.int8))
        record(name, yb, time.time() - t)
    except Exception as e:
        print(name, "FAILED:", e)
    json.dump(log, open(OUT / "summary.json", "w"), indent=1)
print("ALL DONE")

{'method': 'random_over', 'rows': 242568, 'normal': 121284, 'attack': 121284, 'attack_share': 0.5, 'seconds': 2.3}
{'method': 'random_under', 'rows': 38054, 'normal': 19027, 'attack': 19027, 'attack_share': 0.5, 'seconds': 0.4}
{'method': 'smote', 'rows': 242568, 'normal': 121284, 'attack': 121284, 'attack_share': 0.5, 'seconds': 4.7}
{'method': 'borderline_smote', 'rows': 242568, 'normal': 121284, 'attack': 121284, 'attack_share': 0.5, 'seconds': 17.2}
{'method': 'adasyn', 'rows': 242847, 'normal': 121563, 'attack': 121284, 'attack_share': 0.4994, 'seconds': 18.3}
{'method': 'under_then_smote', 'rows': 76108, 'normal': 38054, 'attack': 38054, 'attack_share': 0.5, 'seconds': 2.8}
{'method': 'smote_tomek', 'rows': 242092, 'normal': 121046, 'attack': 121046, 'attack_share': 0.5, 'seconds': 280.1}
{'method': 'smote_enn', 'rows': 236452, 'normal': 118548, 'attack': 117904, 'attack_share': 0.4986, 'seconds': 280.6}
{'method': 'tomek_links', 'rows': 140193, 'normal': 19027, 'attack': 121166,

In [5]:
d = np.load("processed/balanced/smote.npz")
X_bal, y_bal = d["X"], d["y"]
X_val = np.load("processed/X_val.npy"); y_val = np.load("processed/y_val.npy")   # always unbalanced